# AgriCL-Net, step 4: continual learning (learning new diseases without forgetting)

The 112 seen diseases are split **by crop into 5 rounds** (58 diseases first, then 4 rounds of 13-14).
The classifier learns round by round and, after each round, must recognise **every disease seen so far**
without being told which round an image came from.

Three methods start from the same round-0 model:

| Method | Trains each new round on | Role |
| --- | --- | --- |
| `finetune` | new round's images only | lower bound: shows forgetting |
| `replay` | + 20 saved images per old disease (herding) | common baseline |
| `replay_kd` | + knowledge distillation from the previous model + weight aligning | **AgriCL-Net** |

The upper bound is the classifier trained on all diseases at once (notebook 03).

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Run once with `SMOKE_TEST = True`
(3 images per disease, 1 epoch per round), then set it to `False` and choose Runtime > Run all.
Progress is saved after every round, so a disconnect loses at most one round.

## Settings

In [ ]:
MODEL = 'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k'
N_ROUNDS = 5
BASE_EPOCHS = 15     # round 0
INC_EPOCHS = 10      # each later round
EXEMPLARS = 20       # saved images per disease (replay methods)
BATCH = 32
SMOKE_TEST = True    # True: 3 images per disease, 1 epoch per round. Set False for the real run.

## GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Storage and project code from GitHub

Works on **Colab** and **Kaggle**:
* Colab: everything is kept in Google Drive, `MyDrive/Colab Notebooks/agricl-net`.
* Kaggle: turn on *Settings > Accelerator > GPU T4* and *Settings > Internet*. Results go to
  `/kaggle/working/agricl-net` and are kept when you *Save Version*. Downloads go to `/tmp` (fetched again each
  session). To use an earlier notebook's results (or resume a run), attach that notebook's output with
  *Add Input > Notebook Output*: it is copied in automatically below.

In [ ]:
import os, sys, glob, shutil
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    WORK = '/tmp'                               # scratch space: code, unzipped data
    DRIVE = '/kaggle/working/agricl-net'        # results, saved with each notebook version
    CACHE = '/tmp/agricl-cache'                 # dataset zip and SAM2 checkpoint
    for src in glob.glob('/kaggle/input/**/agricl-net/runs/*', recursive=True):
        dst = f'{DRIVE}/runs/{os.path.basename(src)}'
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print('copied earlier results:', os.path.basename(src))
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content'
    DRIVE = '/content/drive/MyDrive/Colab Notebooks/agricl-net'   # dataset zip, SAM2 checkpoint and all results
    CACHE = f'{DRIVE}/cache'
os.makedirs(CACHE, exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)

CODE = f'{WORK}/AGRICL-NET'
if os.path.exists(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q https://github.com/Dev02005/AGRICL-NET.git {CODE}
sys.path.insert(0, CODE)
!git -C {CODE} log -1 --format='code version: %h %s'

## PlantSeg v7 dataset (1.1 GB; on Colab downloaded to Drive once)

In [ ]:
import hashlib
ZIP = f'{CACHE}/plantseg.zip'
if not os.path.exists(ZIP):
    !wget -q --show-progress -O "{ZIP}" "https://zenodo.org/records/17719108/files/plantseg.zip?download=1"

h = hashlib.md5()
with open(ZIP, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 24), b''):
        h.update(chunk)
assert h.hexdigest() == '9358a66dff88cdd15c4fe009763c40a3', f'bad download, md5 {h.hexdigest()}: delete {ZIP} and rerun'

DATA = f'{WORK}/data/plantseg'
if not os.path.exists(f'{DATA}/Metadata.csv'):
    !unzip -q "{ZIP}" -d "{WORK}/data"

In [ ]:
!pip -q install timm
RUN = 'continual_' + MODEL.split('.')[0] + ('_smoke' if SMOKE_TEST else '')
OUT = f'{DRIVE}/runs/{RUN}'
IMG_CACHE = f'{WORK}/plantseg_640'   # resized copies on local disk: much faster than decoding full photos
print('results ->', OUT)

## Run all rounds for all three methods (resumes automatically)

In [ ]:
from agricl.cl.train_cl import CLConfig, run

cfg = CLConfig(
    data_root=DATA, out_dir=OUT, cache_dir=IMG_CACHE, model_name=MODEL, batch_size=BATCH,
    n_tasks=N_ROUNDS,
    base_epochs=1 if SMOKE_TEST else BASE_EPOCHS,
    inc_epochs=1 if SMOKE_TEST else INC_EPOCHS,
    exemplars_per_class=2 if SMOKE_TEST else EXEMPLARS,
    limit_per_class=3 if SMOKE_TEST else 0,
    workers=2,
)
results = run(cfg)

## Results

* **Average incremental accuracy**: mean accuracy over all rounds (higher is better).
* **Final accuracy**: accuracy on all 112 diseases after the last round.
* **Forgetting**: how much accuracy earlier rounds lost by the end (lower is better).
* **Unseen flagged**: share of the three held-out diseases answered "Unknown".

In [ ]:
import json, os, numpy as np, matplotlib.pyplot as plt

rounds = json.load(open(f'{OUT}/rounds.json'))
seen_counts = np.cumsum([len(r['diseases']) for r in rounds])

joint = None
JOINT = f"{DRIVE}/runs/classifier_{MODEL.split('.')[0]}/results.json"
if os.path.exists(JOINT):
    joint = json.load(open(JOINT))['test_seen']['top1']

print(f"{'method':<11} {'avg incr. acc':>13} {'final acc':>10} {'forgetting':>11} {'unseen flagged':>15}")
for m, r in results.items():
    print(f"{m:<11} {r['average_incremental_accuracy']:>13.3f} {r['final_accuracy']:>10.3f} "
          f"{r['forgetting']:>11.3f} {r['calibration']['unseen_flagged_unknown']:>15.3f}")
if joint is not None:
    print(f"{'joint':<11} {'':>13} {joint:>10.3f}   (upper bound, notebook 03)")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for m, r in results.items():
    ax[0].plot(seen_counts[:len(r['step_accuracy'])], r['step_accuracy'], marker='o', label=m)
if joint is not None:
    ax[0].axhline(joint, color='k', ls='--', label='joint (upper bound)')
ax[0].set_xlabel('diseases learned so far'); ax[0].set_ylabel('accuracy on all of them'); ax[0].legend()
ax[0].set_title('accuracy as new diseases are added')

acc = np.array(results['replay_kd']['accuracy_matrix'], float) if 'replay_kd' in results else None
if acc is not None:
    im = ax[1].imshow(acc, vmin=0, vmax=1, cmap='viridis')
    for i in range(acc.shape[0]):
        for j in range(acc.shape[1]):
            if not np.isnan(acc[i, j]):
                ax[1].text(j, i, f'{acc[i, j]:.2f}', ha='center', va='center', color='w', fontsize=8)
    ax[1].set_xlabel('round whose diseases are tested'); ax[1].set_ylabel('after training round')
    ax[1].set_title('replay_kd: accuracy per round'); fig.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.show()

## Next
* Smoke test passed? Set `SMOKE_TEST = False` and run all cells.
* Results: `MyDrive/Colab Notebooks/agricl-net/runs/<run name>/` (`results.json`, `rounds.json`, final model of each method).
* For the report: `replay_kd` should keep accuracy closer to the joint upper bound and forget less than
  `finetune` and `replay`.